# **Практика МИФИ: Хакатон**
---
## Команда **Тима Тимы**

### **Состав:**
**Лид команды:** Жуков Тимофей

**Участники команды:**
1) Морозов Василий Константинович
2) Малыгин Артем Русланович
3) Староверов Максим Алексеевич
4) Никулин Денис Павлович

---
## **Решение Combined Best Pipeline**

**Гипотезы 1:** дедуп log_mean_strict, стекинг Cat+ET через Ridge, SI clean subset  
**Гипотезы 2:** MAE loss, сырая шкала, StratifiedKFold по SI>8, мультисид, feature selection по importance, клиппинг 98%

**4 сабмита:**
1. `submission_fast.csv` - дефолтные параметры, быстро
2. `submission_si_formula.csv` - SI считается как CC50/IC50
3. `submission_optuna_cat.csv` - после тюнинга CatBoost через Optuna
4. `submission_optuna_full.csv` - после тюнинга Cat+ET через Optuna

---
### **0. Импорт библиотек, константы, извлечение данных**

**Библиотеки**

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics.pairwise import euclidean_distances
from catboost import CatBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

**Константы**

In [ ]:
SEED    = 42
N_FOLDS = 5
N_SEEDS = 5
TARGETS = ['IC50', 'CC50', 'SI']

np.random.seed(SEED)

**Извлечение данных**

In [ ]:
# Загрузка датасетов по предоставленной ссылке
train = pd.read_csv('https://drive.google.com/uc?id=159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl')
test  = pd.read_csv('https://drive.google.com/uc?id=1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l')

train = train.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})
print(f'Train: {train.shape}, Test: {test.shape}')

---
### **1. Feature Engineering**

In [ ]:
def fe_with_groups(df):
    df = df.copy()
    df['logP_per_MW']    = df['MolLogP'] / (df['MolWt'] + 1e-9)
    df['TPSA_per_MW']    = df['TPSA']    / (df['MolWt'] + 1e-9)
    df['ASA_ratio']      = df['LabuteASA'] / (df['TPSA'] + 1e-9)
    df['charge_range']   = df['MaxPartialCharge'] - df['MinPartialCharge']
    df['EState_range']   = df['MaxEStateIndex']   - df['MinEStateIndex']
    df['abs_charge_sum'] = df['MaxAbsPartialCharge'] + df['MinAbsPartialCharge']
    groups = {
        'vsa':   [c for c in df.columns if 'VSA' in c],
        'fr':    [c for c in df.columns if c.startswith('fr_')],
        'chi':   [c for c in df.columns if c.startswith('Chi')],
        'bcut':  [c for c in df.columns if c.startswith('BCUT')],
        'slogp': [c for c in df.columns if c.startswith('SlogP')],
    }
    for name, cols in groups.items():
        if cols:
            df[f'{name}_sum']  = df[cols].sum(axis=1)
            df[f'{name}_mean'] = df[cols].mean(axis=1)
            df[f'{name}_std']  = df[cols].std(axis=1)
            df[f'{name}_max']  = df[cols].max(axis=1)
    return df

In [ ]:
train_fe = fe_with_groups(train)
test_fe  = fe_with_groups(test)

drop_cols    = ['IC50', 'CC50', 'SI', 'index']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]

X_all  = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

var_sel   = VarianceThreshold(threshold=0.0)
var_sel.fit(X_all)
good_cols = [c for c, keep in zip(feature_cols, var_sel.get_support()) if keep]
X_all     = X_all[good_cols].fillna(X_all[good_cols].median())
X_test    = X_test[good_cols].fillna(X_all[good_cols].median())
print(f'Фичей после VarianceThreshold: {X_all.shape[1]}')

---
### **2. Feature Selection через CatBoost importance**

In [ ]:
TOP_N = 30

important_features = set()
for target in TARGETS:
    cb_fs = CatBoostRegressor(
        iterations=300, learning_rate=0.05, depth=6,
        loss_function='MAE', random_seed=SEED, verbose=0
    )
    cb_fs.fit(X_all, train[target])
    fi   = pd.Series(cb_fs.get_feature_importance(), index=X_all.columns)
    top  = fi.nlargest(TOP_N).index.tolist()
    important_features.update(top)
    print(f'{target}: топ-3 - {fi.nlargest(3).index.tolist()}')

selected_cols  = list(important_features)
X_sel          = X_all[selected_cols]
X_test_sel     = X_test[selected_cols]
print(f'\nОтобрано фич: {len(selected_cols)}')

---
### **3. Дедупликация log_mean_strict**

**Ф-ия дедубликации**

In [ ]:
def dedup_log_mean_strict(X, train_df, scaler, threshold=0.01):
    X_s = scaler.transform(X)
    visited, rows_to_keep = set(), []
    new_targets = {t: [] for t in TARGETS}

    for i in range(len(X)):
        if i in visited:
            continue
        dists = euclidean_distances(X_s[i:i+1], X_s)[0]
        group = np.where(dists < threshold)[0]
        for idx in group:
            visited.add(idx)
        rows_to_keep.append(i)
        for t in TARGETS:
            vals = train_df[t].iloc[group]
            # log_mean: усредняем в лог-пространстве → expm1 обратно в сырую шкалу
            new_targets[t].append(np.expm1(np.log1p(vals).mean()))

    X_d   = X.iloc[rows_to_keep].reset_index(drop=True)
    Y_raw = {t: pd.Series(new_targets[t]) for t in TARGETS}
    return X_d, Y_raw

In [ ]:
scaler = StandardScaler()
scaler.fit(X_sel)

X_dedup, Y_raw = dedup_log_mean_strict(X_sel, train, scaler)
print(f'Дедуп: {len(X_sel)} → {len(X_dedup)}')

# SI clean subset (обрезаем выбросы)
si_threshold  = train['SI'].quantile(0.98)
clean_mask    = Y_raw['SI'] <= si_threshold
X_clean_si    = X_dedup[clean_mask].reset_index(drop=True)
Y_raw_si      = Y_raw['SI'][clean_mask].reset_index(drop=True)

# Стратификационные метки - считаем отдельно для полного и clean subset
is_good_drug    = (Y_raw['SI'] > 8).astype(int)
is_good_drug_si = (Y_raw_si > 8).astype(int)

# Клиппинг лимиты
clip_limits = {
    'IC50': train['IC50'].quantile(0.98),
    'CC50': train['CC50'].quantile(0.98),
    'SI':   train['SI'].quantile(0.98),
}
print(f'SI clean subset: {len(X_clean_si)}')
print(f'Хороших (SI>8): деdup={is_good_drug.sum()}, si_clean={is_good_drug_si.sum()}')

---
### **4. Функция обучения: мультисид + StratifiedKFold + стекинг**

**Стандартные параметры**

In [ ]:
# Дефолтные параметры - заменятся после Optuna
CAT_PARAMS_DEFAULT = {
    'iterations': 1500, 'learning_rate': 0.03, 'depth': 6,
    'l2_leaf_reg': 7, 'loss_function': 'MAE', 'eval_metric': 'RMSE',
    'verbose': 0
}

ET_PARAMS_DEFAULT = {
    'n_estimators': 1000, 'max_depth': 10, 'min_samples_leaf': 4,
    'max_features': 0.38, 'n_jobs': -1
}

**Ф-ия обучения**

In [ ]:
def train_oof_multiseed(X, y_raw, X_test, strat_labels,
                        cat_params=None, et_params=None,
                        n_seeds=N_SEEDS):
    """
    Обучает Cat и ET с мультисидом, стекирует через Ridge.
    Возвращает: oof_stack, test_stack, rmse_cat, rmse_et, rmse_stack
    """
    cp = cat_params or CAT_PARAMS_DEFAULT
    ep = et_params  or ET_PARAMS_DEFAULT

    oof_cat  = np.zeros(len(X))
    oof_et   = np.zeros(len(X))
    test_cat = np.zeros(len(X_test))
    test_et  = np.zeros(len(X_test))

    for seed in range(n_seeds):
        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                              random_state=SEED + seed * 100)
        oof_cat_s  = np.zeros(len(X))
        oof_et_s   = np.zeros(len(X))
        test_cat_s = np.zeros(len(X_test))
        test_et_s  = np.zeros(len(X_test))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X, strat_labels)):
            X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
            y_tr, y_val = y_raw.iloc[tr_idx], y_raw.iloc[val_idx]

            # CatBoost
            cat_p = dict(cp)
            cat_p['random_seed'] = SEED + seed * 100 + fold
            cat = CatBoostRegressor(**cat_p)
            cat.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                    early_stopping_rounds=150, verbose=0)
            oof_cat_s[val_idx]  = cat.predict(X_val)
            test_cat_s         += cat.predict(X_test) / N_FOLDS

            # ExtraTrees
            et_p = dict(ep)
            et_p['random_state'] = SEED + seed * 100 + fold
            et = ExtraTreesRegressor(**et_p)
            et.fit(X_tr, y_tr)
            oof_et_s[val_idx]  = et.predict(X_val)
            test_et_s         += et.predict(X_test) / N_FOLDS

        oof_cat  += oof_cat_s  / n_seeds
        oof_et   += oof_et_s   / n_seeds
        test_cat += test_cat_s / n_seeds
        test_et  += test_et_s  / n_seeds

    # Ridge стекинг
    meta = Ridge(alpha=1.0)
    meta.fit(np.column_stack([oof_cat, oof_et]), y_raw)
    oof_stack  = meta.predict(np.column_stack([oof_cat,  oof_et]))
    test_stack = meta.predict(np.column_stack([test_cat, test_et]))

    rmse_cat   = np.sqrt(mean_squared_error(y_raw, oof_cat))
    rmse_et    = np.sqrt(mean_squared_error(y_raw, oof_et))
    rmse_stack = np.sqrt(mean_squared_error(y_raw, oof_stack))

    return oof_stack, test_stack, rmse_cat, rmse_et, rmse_stack

**Ф-ия сохранения результата**

In [ ]:
def make_submission(test_ic50, test_cc, test_si, filename):
    sub = pd.DataFrame()
    sub['index'] = test['index'] if 'index' in test.columns else test.index
    sub['IC50']  = np.maximum(0.001, np.minimum(test_ic50, clip_limits['IC50']))
    sub['CC50']  = np.maximum(0.001, np.minimum(test_cc,   clip_limits['CC50']))
    sub['SI']    = np.maximum(0.001, np.minimum(test_si,   clip_limits['SI']))
    sub.to_csv(filename, index=False)
    print(f'✓ {filename}')
    return sub

---
### **5.1. Сабмит 1: быстрый (дефолтные параметры)**

In [ ]:
print('=== IC50 ===')
oof_ic50, test_ic50, r_cat, r_et, r_ic50 = train_oof_multiseed(
    X_dedup, Y_raw['IC50'], X_test_sel, is_good_drug)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_ic50:.2f}')

print('=== CC50 ===')
oof_cc, test_cc, r_cat, r_et, r_cc = train_oof_multiseed(
    X_dedup, Y_raw['CC50'], X_test_sel, is_good_drug)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_cc:.2f}')

print('=== SI ===')
oof_si, test_si, r_cat, r_et, r_si = train_oof_multiseed(
    X_clean_si, Y_raw_si, X_test_sel, is_good_drug_si)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_si:.2f}')

print(f'\n Mean RMSE: {np.mean([r_ic50, r_cc, r_si]):.2f}')

sub1 = make_submission(test_ic50, test_cc, test_si, 'submission_fast.csv')

---
### 5.2. **Сабмит 2: SI как формула CC50/IC50**

In [ ]:
test_si_formula = test_cc / (test_ic50 + 1e-9)

# OOF метрика для формульного SI (на дедуп данных)
oof_si_formula = oof_cc / (oof_ic50 + 1e-9)
# Считаем на чистом subset чтобы сравнить честно
r_si_formula = np.sqrt(mean_squared_error(
    Y_raw['CC50'][clean_mask].values / (Y_raw['IC50'][clean_mask].values + 1e-9),
    oof_cc[clean_mask] / (oof_ic50[clean_mask] + 1e-9)
))
print(f'SI formula RMSE (OOF): {r_si_formula:.2f}')
print(f'SI model  RMSE (OOF): {r_si:.2f}')

sub2 = make_submission(test_ic50, test_cc, test_si_formula, 'submission_si_formula.csv')

---
### **5.3. Optuna: тюним Cat и ET**

In [ ]:
def optuna_objective(trial, model_type='cat', n_seeds=2):
    """
    Быстрая оценка: 2 сида вместо 5, без стекинга.
    Усредняем RMSE по трём таргетам.
    """
    if model_type == 'cat':
        bp = trial.suggest_categorical('bootstrap_type',
                                       ['Bayesian', 'Bernoulli', 'MVS'])
        params = {
            'iterations':      trial.suggest_int('iterations', 500, 2000),
            'depth':           trial.suggest_int('depth', 4, 10),
            'learning_rate':   trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'l2_leaf_reg':     trial.suggest_float('l2_leaf_reg', 1, 15),
            'random_strength': trial.suggest_float('random_strength', 0.1, 20),
            'border_count':    trial.suggest_int('border_count', 32, 255),
            'bootstrap_type':  bp,
            'loss_function':   'MAE',
            'eval_metric':     'RMSE',
            'verbose': 0
        }
        if bp == 'Bayesian':
            params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0, 10)
        else:
            params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
    else:
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 500, 2000),
            'max_depth':         trial.suggest_int('max_depth', 5, 25),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'max_features':      trial.suggest_float('max_features', 0.1, 1.0),
            'n_jobs': -1
        }

    rmses = []
    datasets = [
        (X_dedup,    Y_raw['IC50'], is_good_drug),
        (X_dedup,    Y_raw['CC50'], is_good_drug),
        (X_clean_si, Y_raw_si,     is_good_drug_si),
    ]

    for X_t, y_t, strat in datasets:
        oof = np.zeros(len(X_t))
        for seed in range(n_seeds):
            skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                  random_state=SEED + seed * 100)
            oof_s = np.zeros(len(X_t))
            for fold, (tr_idx, val_idx) in enumerate(skf.split(X_t, strat)):
                X_tr, X_val = X_t.iloc[tr_idx], X_t.iloc[val_idx]
                y_tr, y_val = y_t.iloc[tr_idx], y_t.iloc[val_idx]
                if model_type == 'cat':
                    p = dict(params)
                    p['random_seed'] = SEED + seed * 100 + fold
                    m = CatBoostRegressor(**p)
                    m.fit(X_tr, y_tr, eval_set=(X_val, y_val),
                          early_stopping_rounds=100, verbose=0)
                else:
                    p = dict(params)
                    p['random_state'] = SEED + seed * 100 + fold
                    m = ExtraTreesRegressor(**p)
                    m.fit(X_tr, y_tr)
                oof_s[val_idx] = m.predict(X_val)
            oof += oof_s / n_seeds
        rmses.append(np.sqrt(mean_squared_error(y_t, oof)))

    return np.mean(rmses)

In [ ]:
N_TRIALS_CAT = 80
N_TRIALS_ET  = 60

print(f'Optuna CatBoost ({N_TRIALS_CAT} trials)...')
study_cat = optuna.create_study(direction='minimize',
                                pruner=optuna.pruners.MedianPruner())
study_cat.optimize(lambda t: optuna_objective(t, 'cat'),
                   n_trials=N_TRIALS_CAT, show_progress_bar=True)

best_cat = study_cat.best_params
best_cat.update({'loss_function': 'MAE', 'eval_metric': 'RMSE', 'verbose': 0})
print(f'\nCat best RMSE: {study_cat.best_value:.2f}')
print(best_cat)

print(f'\nOptuna ExtraTrees ({N_TRIALS_ET} trials)...')
study_et = optuna.create_study(direction='minimize')
study_et.optimize(lambda t: optuna_objective(t, 'et'),
                  n_trials=N_TRIALS_ET, show_progress_bar=True)

best_et = study_et.best_params
best_et.update({'n_jobs': -1})
print(f'\nET best RMSE: {study_et.best_value:.2f}')
print(best_et)

---
### **5.4. Сабмит 3: только Optuna Cat (ET дефолт)**

In [ ]:
print('=== IC50 (optuna cat) ===')
oof_ic50_o, test_ic50_o, r_cat, r_et, r_ic50_o = train_oof_multiseed(
    X_dedup, Y_raw['IC50'], X_test_sel, is_good_drug,
    cat_params=best_cat)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_ic50_o:.2f}')

print('=== CC50 (optuna cat) ===')
oof_cc_o, test_cc_o, r_cat, r_et, r_cc_o = train_oof_multiseed(
    X_dedup, Y_raw['CC50'], X_test_sel, is_good_drug,
    cat_params=best_cat)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_cc_o:.2f}')

print('=== SI (optuna cat) ===')
oof_si_o, test_si_o, r_cat, r_et, r_si_o = train_oof_multiseed(
    X_clean_si, Y_raw_si, X_test_sel, is_good_drug_si,
    cat_params=best_cat)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_si_o:.2f}')

print(f'\nMean RMSE (optuna cat): {np.mean([r_ic50_o, r_cc_o, r_si_o]):.2f}')
sub3 = make_submission(test_ic50_o, test_cc_o, test_si_o, 'submission_optuna_cat.csv')

---
### **5.5. Сабмит 4: Optuna Cat + Optuna ET**

In [ ]:
print('=== IC50 (optuna cat+et) ===')
oof_ic50_f, test_ic50_f, r_cat, r_et, r_ic50_f = train_oof_multiseed(
    X_dedup, Y_raw['IC50'], X_test_sel, is_good_drug,
    cat_params=best_cat, et_params=best_et)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_ic50_f:.2f}')

print('=== CC50 (optuna cat+et) ===')
oof_cc_f, test_cc_f, r_cat, r_et, r_cc_f = train_oof_multiseed(
    X_dedup, Y_raw['CC50'], X_test_sel, is_good_drug,
    cat_params=best_cat, et_params=best_et)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_cc_f:.2f}')

print('=== SI (optuna cat+et) ===')
oof_si_f, test_si_f, r_cat, r_et, r_si_f = train_oof_multiseed(
    X_clean_si, Y_raw_si, X_test_sel, is_good_drug_si,
    cat_params=best_cat, et_params=best_et)
print(f'  cat={r_cat:.2f}  et={r_et:.2f}  stack={r_si_f:.2f}')

print(f'\nMean RMSE (optuna full): {np.mean([r_ic50_f, r_cc_f, r_si_f]):.2f}')
sub4 = make_submission(test_ic50_f, test_cc_f, test_si_f, 'submission_optuna_full.csv')

---
### **6. Итог**

In [ ]:
# Итоговое сравнение
print('\n=== Итог ===')
print(f'Fast:        {np.mean([r_ic50,   r_cc,   r_si]):.2f}')
print(f'Optuna cat:  {np.mean([r_ic50_o, r_cc_o, r_si_o]):.2f}')
print(f'Optuna full: {np.mean([r_ic50_f, r_cc_f, r_si_f]):.2f}')